# Практикум 11. Расстояния, сходство, центроиды

Сопровождает интерактивную лабораторию №11 («Расстояния и центроид»). Тема курса: математические основы кластеризации текстов

Ноутбук самодостаточен: первая ячейка создаёт файл `common.py` с учебным корпусом и вспомогательными функциями. Запускайте ячейки сверху вниз. Корпус учебный (шаблонный), для реальной задачи подставьте свой CSV со столбцами `text` и `label`.

In [ ]:
%%writefile common.py
"""Общие функции для практикумов по классификации и кластеризации текстов."""
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

TOPICS = {
    "спорт": ("матч команда гол игрок тренер чемпионат турнир победа поражение сезон стадион болельщик судья счёт "
              "защитник нападающий вратарь кубок лига финал трансфер клуб удар пенальти соревнование спортсмен "
              "рекорд медаль олимпиада тренировка ничья").split(),
    "политика": ("закон депутат выборы партия голосование президент министр правительство парламент реформа бюджет "
                 "санкции кандидат коалиция министерство указ поправка законопроект оппозиция референдум дипломатия "
                 "мэр губернатор конституция заседание комиссия переговоры саммит избиратель мандат полномочия").split(),
    "кулинария": ("суп рецепт соль мясо готовить духовка овощи сковорода кастрюля тесто сахар специи варить жарить "
                  "запекать соус бульон салат повар блюдо ингредиент масло мука яйцо молоко чеснок лук перец "
                  "картофель десерт выпечка начинка маринад").split(),
    "технологии": ("процессор программа приложение смартфон нейросеть алгоритм данные сервер обновление разработчик "
                   "компьютер память экран камера браузер сеть интернет облако код платформа пользователь устройство "
                   "чип батарея ноутбук софт видеокарта система интерфейс шифрование").split(),
}
NEUTRAL = ("сегодня очень новый говорят после который также можно было стал сообщили вчера недавно ещё всё "
           "несколько важный первый").split()


def make_corpus(n_per_class=150, noise=0.3, length=(6, 11), seed=0):
    """Учебный корпус из коротких 'документов' четырёх тем.
    noise: доля слов, взятых не из своей темы (нейтральные или чужие). Чем больше noise, тем труднее задача."""
    rng = random.Random(seed)
    names = list(TOPICS)
    rows = []
    for label in names:
        others = [t for t in names if t != label]
        for _ in range(n_per_class):
            words = []
            for _ in range(rng.randint(*length)):
                u = rng.random()
                if u < noise * 0.5:
                    words.append(rng.choice(NEUTRAL))
                elif u < noise:
                    words.append(rng.choice(TOPICS[rng.choice(others)]))
                else:
                    words.append(rng.choice(TOPICS[label]))
            rows.append((" ".join(words), label))
    df = pd.DataFrame(rows, columns=["text", "label"])
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def project_2d(X, seed=0):
    """Проекция разреженной матрицы TF-IDF на плоскость (TruncatedSVD)."""
    from sklearn.decomposition import TruncatedSVD
    return TruncatedSVD(2, random_state=seed).fit_transform(X)


def scatter2d(Z, labels, title="", ax=None):
    ax = ax or plt.gca()
    codes, uniq = pd.factorize(pd.Series(labels))
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab10", s=14, alpha=0.8)
    ax.set_title(title)
    handles = [plt.Line2D([], [], marker="o", ls="", color=sc.cmap(sc.norm(i)), label=str(u)) for i, u in enumerate(uniq)]
    ax.legend(handles=handles, fontsize=8)
    return ax


def top_terms(coef, terms, class_names, n=8):
    """Печатает слова с наибольшими весами для каждого класса (линейные модели)."""
    terms = np.asarray(terms)
    for i, name in enumerate(class_names):
        idx = np.argsort(coef[i])[::-1][:n]
        print(f"{name:12s}", ", ".join(terms[idx]))

In [ ]:
from common import *

# Учебный корпус: 4 темы. noise - доля "посторонних" слов (чем больше, тем труднее).
df = make_corpus(n_per_class=150, noise=0.3, seed=0)
# Свой корпус вместо учебного: df = pd.read_csv("texts.csv")  # столбцы: text, label
print(df.label.value_counts().to_dict())
df.head()

## 1. Матрицы сходства документов

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

sample = pd.concat([df[df.label == l].head(3) for l in sorted(df.label.unique())]).reset_index(drop=True)
X = TfidfVectorizer().fit_transform(sample.text)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for a, M, t, cm in [(ax[0], cosine_similarity(X), "Косинусное сходство", "Blues"), (ax[1], euclidean_distances(X), "Евклидово расстояние", "Blues_r")]:
    im = a.imshow(M, cmap=cm); a.set_title(t); plt.colorbar(im, ax=a)
    a.set_xticks(range(len(sample))); a.set_xticklabels(sample.label, rotation=90); a.set_yticks(range(len(sample))); a.set_yticklabels(sample.label); a.grid(False)
plt.tight_layout(); plt.show()

## 2. Длина документа: косинус не меняется, евклид меняется

In [ ]:
cv = CountVectorizer()
cv.fit(["матч команда гол игрок закон депутат"])
a = cv.transform(["матч команда гол"]); b = cv.transform(["матч команда гол матч команда гол"])
print("Косинусное сходство:", cosine_similarity(a, b)[0, 0].round(3))
print("Евклидово расстояние:", euclidean_distances(a, b)[0, 0].round(3))

## 3. Центроиды классов и классификация по ближайшему центроиду

In [ ]:
vec = TfidfVectorizer(); Xall = vec.fit_transform(df.text)
classes = sorted(df.label.unique())
cent = np.vstack([np.asarray(Xall[(df.label == c).values].mean(axis=0)) for c in classes])
pred = np.array(classes)[cosine_similarity(Xall, cent).argmax(axis=1)]
print("Accuracy по ближайшему центроиду:", round((pred == df.label.values).mean(), 3))
terms = np.array(vec.get_feature_names_out())
for c, row in zip(classes, cent):
    print(f"{c:12s}", ", ".join(terms[row.argsort()[::-1][:6]]))

## Задания

1. Замените TF-IDF на `CountVectorizer`. Как изменились матрицы сходства?
2. Добавьте в корпус «выбросы» (очень длинные тексты) и сравните положение центроида с покоординатной медианой.
3. Докажите численно: для L2-нормированных векторов `d² = 2·(1 − cos)`.